# IEX vs Okta Checker
So sánh trạng thái agent trong Okta snapshot với lịch IEX (dữ liệu đã qua cleaner).

In [9]:
import pandas as pd
from datetime import datetime

# Hiển thị đủ số dòng và số cột mong muốn
pd.set_option("display.max_rows", 100)   # tối đa số dòng hiển thị
pd.set_option("display.max_columns", None)  # hiện tất cả các cột
pd.set_option("display.width", None)   # không giới hạn độ rộng

In [10]:
# Đọc dữ liệu từ file đã clean (iex_cleaner xuất ra)
iex_df = pd.read_excel('iex-data-extracted.xlsx')
okta_df = pd.read_csv('okta.csv')

# Xóa cột "Available On" nếu tồn tại
if "Available On" in okta_df.columns:
    okta_df = okta_df.drop(columns=["Available On"])

iex_df.head(59)

,IEX Id,Name,Shift,Date,Activity,Start time,End time,Site,Supervisor Name,LOB,Email Id
0,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,2025-08-26 13:00:00,2025-08-26 13:45:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
1,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Break,2025-08-26 13:45:00,2025-08-26 14:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
2,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,2025-08-26 14:00:00,2025-08-26 17:45:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
3,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Break,2025-08-26 17:45:00,2025-08-26 18:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
4,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,2025-08-26 18:00:00,2025-08-26 20:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
5,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Lunch,2025-08-26 20:00:00,2025-08-26 21:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
6,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Open Time,2025-08-26 21:00:00,2025-08-26 22:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
7,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Open Time,2025-08-27 13:00:00,2025-08-27 14:10:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
8,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Break,2025-08-27 14:10:00,2025-08-27 14:25:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
9,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-27,Open Time,2025-08-27 14:25:00,2025-08-27 16:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com


In [11]:
# Đổi tên cột Okta cho đồng bộ (nếu cần chỉnh lại tuỳ dataset)
okta_df = okta_df.rename(columns={
    'userName': 'Name',
    'status': 'Activity',
    'duration': 'Duration'
})
okta_df['CheckTime'] = datetime.now()
okta_df.head()

,Agent Name,Duration,State,Assigned Workitem Count,Agent Email,Queue Group / Routing Profile,Forecast Group,Manager Email,Business Location,CheckTime
0,"Bui, Ngoc Thuan Vy",00:00:27,AVAILABLECHAT,1.0,ngocthuanvy.bui@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-27 03:42:07.703298
1,"Bui, The Anh",00:05:24,AVAILABLECHAT,NaN,theanh.bui1@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-27 03:42:07.703298
2,"Bui, Thi Ngoc Tram",00:08:31,AVAILABLECHAT,1.0,thingoctram.bui@concentrix.com,Chat_OD_EN_Car_Activity,GEN_GEN_EN_GCS_GLG_CHT,chihuy.vong@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-27 03:42:07.703298
3,"Chau, Thien Kim",02:58:44,LOGIN,NaN,thienkim.chau@concentrix.com,Chat_OD_EN_Dual_GDS,GEN_GEN_EN_GCS_GNL_CHT,kirpan.patar@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-27 03:42:07.703298
4,"Chinh, Ngoc Thu",00:18:52,AVAILABLECHAT,NaN,ngocthu.chinh@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,nguyenthaonhi.tran@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-27 03:42:07.703298


In [12]:
# Chuyển tất cả giá trị "Open Time" trong cột Activity thành "AVAILABLECHAT"
iex_df["Activity"] = iex_df["Activity"].replace("Open Time", "AVAILABLECHAT")

# Chuyển Start/End về datetime
iex_df['Start time'] = pd.to_datetime(iex_df['Start time'], errors='coerce')
iex_df['End time']   = pd.to_datetime(iex_df['End time'], errors='coerce')
iex_df.head()

,IEX Id,Name,Shift,Date,Activity,Start time,End time,Site,Supervisor Name,LOB,Email Id
0,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,AVAILABLECHAT,2025-08-26 13:00:00,2025-08-26 13:45:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
1,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Break,2025-08-26 13:45:00,2025-08-26 14:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
2,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,AVAILABLECHAT,2025-08-26 14:00:00,2025-08-26 17:45:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
3,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,Break,2025-08-26 17:45:00,2025-08-26 18:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com
4,3052306,"BUI, BADUONG",1:00 PM - 10:00 PM,2025-08-26,AVAILABLECHAT,2025-08-26 18:00:00,2025-08-26 20:00:00,ONEHUB,To Anh Duy,Lodging,baduong.bui@concentrix.com


In [13]:
import re

def clean_name(name: str) -> str:
    if pd.isna(name):
        return name
    # Thay dấu phẩy bằng khoảng trắng
    name = name.replace(",", " ")
    # Thêm khoảng trắng trước chữ in hoa (trừ chữ cái đầu)
    name = re.sub(r'(?<!^)(?=[A-Z])', ' ', name)
    # Chuẩn hoá khoảng trắng thừa
    name = " ".join(name.split())
    return name.strip()

# Chuẩn hoá cho cả IEX và Okta
iex_df["Name"] = iex_df["Name"].astype(str).map(clean_name)
okta_df["Agent Name"] = okta_df["Agent Name"].astype(str).map(clean_name)
#iex_df.head()
okta_df.head()

,Agent Name,Duration,State,Assigned Workitem Count,Agent Email,Queue Group / Routing Profile,Forecast Group,Manager Email,Business Location,CheckTime
0,Bui Ngoc Thuan Vy,00:00:27,AVAILABLECHAT,1.0,ngocthuanvy.bui@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-27 03:42:07.703298
1,Bui The Anh,00:05:24,AVAILABLECHAT,NaN,theanh.bui1@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,thienkim.chau@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-27 03:42:07.703298
2,Bui Thi Ngoc Tram,00:08:31,AVAILABLECHAT,1.0,thingoctram.bui@concentrix.com,Chat_OD_EN_Car_Activity,GEN_GEN_EN_GCS_GLG_CHT,chihuy.vong@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-27 03:42:07.703298
3,Chau Thien Kim,02:58:44,LOGIN,NaN,thienkim.chau@concentrix.com,Chat_OD_EN_Dual_GDS,GEN_GEN_EN_GCS_GNL_CHT,kirpan.patar@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-27 03:42:07.703298
4,Chinh Ngoc Thu,00:18:52,AVAILABLECHAT,NaN,ngocthu.chinh@concentrix.com,Chat_OD_EN_Lodging,GEN_GEN_EN_GCS_GLG_CHT,nguyenthaonhi.tran@concentrix.com,Concentrix (Ho Chi Minh City),2025-08-27 03:42:07.703298


In [14]:
# Hàm chuẩn hoá tên để so sánh
def normalize_name(name: str) -> str:
    return ''.join(str(name).split()).upper()

# 1. So sánh giữa Okta và IEX
results = []
now = datetime.now()

for _, row in okta_df.iterrows():
    name_okta = row['Agent Name']
    activity_okta = row['State']
    workitem_count = row.get('Assigned Workitem Count', None)  # lấy cột này, tránh lỗi nếu cột không tồn tại
    
    # Skip các trường hợp không phải Available chat nhưng productive
    if str(activity_okta).strip().upper() != "AVAILABLECHAT" and pd.notna(workitem_count):
        continue  # bỏ qua agent này, sang vòng lặp tiếp theo
    
    # Tìm agent trong IEX (dùng normalize_name để so sánh)
    iex_agent = iex_df[iex_df['Name'].apply(normalize_name) == normalize_name(name_okta)]
    iex_now = iex_agent[(iex_agent['Start time'] <= now) & (iex_agent['End time'] >= now)]
    
    if not iex_now.empty:
        activity_iex = iex_now.iloc[0]['Activity']
        start_iex = iex_now.iloc[0]['Start time']
        end_iex = iex_now.iloc[0]['End time']
        site = iex_now.iloc[0].get('Site', None)
        lob = iex_now.iloc[0].get('LOB', None)
        supervisor = iex_now.iloc[0].get('Supervisor Name', None)
    else:
        activity_iex = 'N/A'
        start_iex = None
        end_iex = None
        site = None
        lob = None
        supervisor = None
    
    results.append({
        'Agent': name_okta,   # giữ nguyên format tên từ Okta
        'LOB': lob,
        'Site': site,
        'Supervisor Name': supervisor,
        'Activity_Okta': activity_okta,
        'Activity_IEX': activity_iex,
        'Start_IEX': start_iex,
        'End_IEX': end_iex,
        'Match': str(activity_okta).strip().lower() == str(activity_iex).strip().lower()
    })


# 2. Bổ sung agent đang CÓ MẶT trong IEX (any activity hợp lệ) nhưng KHÔNG có trong Okta
iex_current = iex_df[
    (iex_df['Start time'] <= now) & 
    (iex_df['End time'] >= now)
].copy()

# Chỉ lấy những dòng có activity hợp lệ
iex_current = iex_current[iex_current['Activity'].notna() & (iex_current['Activity'].astype(str).str.strip() != '')]

# Nếu 1 agent có nhiều dòng tại thời điểm now, giữ dòng Start sớm nhất
iex_current = iex_current.sort_values(['Name', 'Start time']).drop_duplicates(subset=['Name'], keep='first')

# Danh sách agent đã có trong Okta (normalize để so sánh)
okta_agents_norm = set(okta_df['Agent Name'].apply(normalize_name))

for _, row in iex_current.iterrows():
    if normalize_name(row['Name']) not in okta_agents_norm:
        results.append({
            'Agent': row['Name'],  # giữ nguyên format tên từ IEX
            'LOB': row.get('LOB', None),
            'Site': row.get('Site', None),
            'Supervisor Name': row.get('Supervisor Name', None),
            'Activity_Okta': 'N/A',
            'Activity_IEX': row['Activity'],
            'Start_IEX': row['Start time'],
            'End_IEX': row['End time'],
            'Match': False
        })

# 3. Kết quả cuối
result_df = pd.DataFrame(results)

# Danh sách activity cần loại bỏ
exclude_activities = ["Termination", "No Call/No Show", "Unpaid Leave", "PTO"]

# Lọc bỏ những dòng có Activity_IEX nằm trong danh sách exclude
result_df = result_df[~result_df['Activity_IEX'].isin(exclude_activities)]

# Sort theo LOB rồi Site
result_df = result_df.sort_values(by=['LOB', 'Site'], ascending=[True, True]).reset_index(drop=True)

result_df

,Agent,LOB,Site,Supervisor Name,Activity_Okta,Activity_IEX,Start_IEX,End_IEX,Match
0,Bui Thi Ngoc Tram,Lodging,FLEMINGTON,Jimi Kurt (Nguyễn Khôi),AVAILABLECHAT,AVAILABLECHAT,2025-08-27 01:55:00,2025-08-27 04:15:00,True
1,Duong Nguyen Hoang Huy,Lodging,FLEMINGTON,Do Hong Hanh,AVAILABLECHAT,AVAILABLECHAT,2025-08-27 01:55:00,2025-08-27 04:00:00,True
2,Ho Ky Duyen,Lodging,FLEMINGTON,Jimi Kurt (Nguyễn Khôi),AVAILABLECHAT,AVAILABLECHAT,2025-08-27 03:20:00,2025-08-27 04:45:00,True
3,Ngo Nguyen Bao Khue,Lodging,FLEMINGTON,Jimi Kurt (Nguyễn Khôi),AVAILABLECHAT,AVAILABLECHAT,2025-08-27 02:15:00,2025-08-27 04:50:00,True
4,Nguyen Hieu Han,Lodging,FLEMINGTON,Jimi Kurt (Nguyễn Khôi),AVAILABLECHAT,AVAILABLECHAT,2025-08-27 01:55:00,2025-08-27 04:05:00,True
5,Nguyen Thi Thanh Tuyen,Lodging,FLEMINGTON,Jimi Kurt (Nguyễn Khôi),AVAILABLECHAT,AVAILABLECHAT,2025-08-27 02:00:00,2025-08-27 03:50:00,True
6,Vo Thuy Vy,Lodging,FLEMINGTON,Do Hong Hanh,LUNCH,AVAILABLECHAT,2025-08-27 03:20:00,2025-08-27 04:20:00,False
7,Bui Ngoc Thuan Vy,Lodging,ONEHUB,Truong Thien Thanh Toan,AVAILABLECHAT,Break,2025-08-27 03:40:00,2025-08-27 03:55:00,False
8,Bui The Anh,Lodging,ONEHUB,Chau Thien Kim,AVAILABLECHAT,AVAILABLECHAT,2025-08-27 03:20:00,2025-08-27 05:00:00,True
9,Chinh Ngoc Thu,Lodging,ONEHUB,Tran Sharon,AVAILABLECHAT,AVAILABLECHAT,2025-08-27 01:40:00,2025-08-27 05:00:00,True


In [15]:
# Xuất mismatch ra file Excel
mismatch_df = result_df[(result_df['Match'] == False) & (result_df["Activity_IEX"] != "N/A")]
mismatch_df.to_excel('iex_okta_mismatch.xlsx', index=False)
mismatch_df[mismatch_df['Activity_IEX'] != "AVAILABLECHAT"]

,Agent,LOB,Site,Supervisor Name,Activity_Okta,Activity_IEX,Start_IEX,End_IEX,Match
7,Bui Ngoc Thuan Vy,Lodging,ONEHUB,Truong Thien Thanh Toan,AVAILABLECHAT,Break,2025-08-27 03:40:00,2025-08-27 03:55:00,False
10,Dang Chau Anh,Lodging,ONEHUB,Truong Thien Thanh Toan,AVAILABLECHAT,Break,2025-08-27 03:35:00,2025-08-27 03:50:00,False
12,Duong Cong Hoang,Lodging,ONEHUB,Tran Sharon,AVAILABLECHAT,Break,2025-08-27 03:35:00,2025-08-27 03:50:00,False
15,Le Hoai Minh Ngan,Lodging,ONEHUB,Chau Thien Kim,AVAILABLECHAT,Break,2025-08-27 03:30:00,2025-08-27 03:45:00,False


In [16]:
# --- Export full comparison with both True/False ---
try:
    out_file = 'iex_okta_comparison.xlsx'
    result_df.to_excel(out_file, index=False)
    print(f'Saved full comparison to: {out_file}')
    result_df
except NameError as e:
    print('result_df is not defined. Please run the comparison cells above first.')
    raise


Saved full comparison to: iex_okta_comparison.xlsx
